<a href="https://colab.research.google.com/github/k90262/ML2022-Spring/blob/demo%2Fbill_testing/A_toy_example_for_HW7_Bert_QA_2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ml2023spring_hw7_path = kagglehub.competition_download('ml2023spring-hw7')

print('Data source import complete.')


# A toy example for HW7 Bert QA

Source code from:  
- [ML2021 hw7 English version](https://www.youtube.com/watch?v=SnqP5HnOBD8)
- [李宏毅ML2021 HW7 BERT-Question Answering_hw7 李宏毅 report questions-CSDN博客](https://blog.csdn.net/qq_39610915/article/details/119913009)

## Install Transfomer

In [ ]:
#!pip install transformers==4.5.0
!pip install transformers

## Import Package

In [ ]:
import torch
from transformers import AdamW, BertTokenizerFast, BertForQuestionAnswering

## Load Model and Tokenizer

In [ ]:
# model_name can be either: models in huggingface model hub or models saved using save_pretrained
model_name = 'bert-base-chinese'
model = BertForQuestionAnswering.from_pretrained(model_name)

In [ ]:
chi_tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')
eng_tokenizer = BertTokenizerFast.from_pretrained('bert-base-cased')

## Tokenize

In [ ]:
chi_paragraph = '李弘毅幾班大金。2021 ML'
tokens = chi_tokenizer.tokenize(chi_paragraph)
print(tokens)
chi_tokenizer.convert_tokens_to_ids(tokens)

In [ ]:
eng_paragraph = 'Lee Hung-yi which class Daikin.'
tokens = eng_tokenizer.tokenize(eng_paragraph)
print(tokens)
eng_tokenizer.convert_tokens_to_ids(tokens)

## Encode vs Decode

In [ ]:
question = '李弘毅幾班?'
paragraph = '李弘毅幾班大金'
encoded = chi_tokenizer.encode(question, paragraph)
decoded = chi_tokenizer.decode(encoded)
print(encoded)
print(decoded)

## Model Inputs

In [ ]:
inputs = chi_tokenizer(question, paragraph, return_tensors='pt')
# Indices of input sequence tokens in the vocabulary
print('Input ids:      ', inputs['input_ids'])
# Segment token indices to indicate first and second portions of the inputs. Indices are selected in [0, 1]
print('Token type ids: ', inputs['token_type_ids'])
# Mask to avoid performing attention on padding token indices. Mask values selected in [0, 1]
print('Attention mask: ', inputs['attention_mask'])

## Testing (Chinese) - Before Training

In [ ]:
question = '李弘毅幾班?'
paragraph = '李弘毅幾班大金。'
inputs = chi_tokenizer(question, paragraph, return_tensors='pt')

with torch.no_grad():
    output = model(**inputs)
# output = model(input_ids=inputs['input_ids'], token_type_ids=inputs['token_type_ids'], attention_mask=inputs['attention_mask'])

print("start_logits: ")
print(output.start_logits)

print("end_logits: ")
print(output.end_logits)

start = torch.argmax(output.start_logits) # 返回dim维度上张量最大值的索引。
end = torch.argmax(output.end_logits)
print("start position: ", start.item()) # 一个元素张量可以用x.item()得到元素值
print("end position:   ", end.item())

# 获取预测的start和end的token的id
predict_id = inputs['input_ids'][0][start : end + 1]
print("predict_id:     ", predict_id)
# 根据id解码出原文
predict_answer = chi_tokenizer.decode(predict_id)
print("predict_answer: ", predict_answer)

## Training (Chinese)
For Question Answering, loss is the sum of cross entropy between model prediction and correct answer

In [ ]:
# 指定正确答案的开始结束位置，13
output = model(**inputs, start_positions=torch.tensor([13]), end_positions=torch.tensor([14]))
print("loss: ", output.loss)

optimizer = AdamW(model.parameters(), lr=1e-4)
output.loss.backward()
optimizer.step()

## Testing (Chinese) - After Training (Code is the same as the code in previous'before training' cell)

In [ ]:
question = '李弘毅幾班?'
paragraph = '李弘毅幾班大金。'
inputs = chi_tokenizer(question, paragraph, return_tensors='pt')

with torch.no_grad():
    output = model(**inputs)
# output = model(input_ids=inputs['input_ids'], token_type_ids=inputs['token_type_ids'], attention_mask=inputs['attention_mask'])

print("start_logits: ")
print(output.start_logits)

print("end_logits: ")
print(output.end_logits)

start = torch.argmax(output.start_logits) # 返回dim维度上张量最大值的索引。
end = torch.argmax(output.end_logits)
print("start position: ", start.item()) # 一个元素张量可以用x.item()得到元素值
print("end position:   ", end.item())

# 获取预测的start和end的token的id
predict_id = inputs['input_ids'][0][start : end + 1]
print("predict_id:     ", predict_id)
# 根据id解码出原文
predict_answer = chi_tokenizer.decode(predict_id)
print("predict_answer: ", predict_answer)

## Testing (English)

In [ ]:
question = 'Why does Jeanie like Tom?'
paragraph = 'Jeanie likes Tom because Tom is good at deep learning.'
inputs = eng_tokenizer(question, paragraph, return_tensors='pt')

with torch.no_grad():
    output = model(**inputs)
# output = model(input_ids=inputs['input_ids'], token_type_ids=inputs['token_type_ids'], attention_mask=inputs['attention_mask'])

print("start_logits: ")
print(output.start_logits)

print("end_logits: ")
print(output.end_logits)

start = torch.argmax(output.start_logits) # 返回dim维度上张量最大值的索引。
end = torch.argmax(output.end_logits)
print("start position: ", start.item()) # 一个元素张量可以用x.item()得到元素值
print("end position:   ", end.item())

# 获取预测的start和end的token的id
predict_id = inputs['input_ids'][0][start : end + 1]
print("predict_id:     ", predict_id)
# 根据id解码出原文
predict_answer = eng_tokenizer.decode(predict_id)
print("predict_answer: ", predict_answer)

## Training (English)
For Question Answering, loss is the sum of cross entropy between model prediction and correct answer

Try to find the position of the anwser (part 1)

In [ ]:
inputs = eng_tokenizer(question, paragraph, return_tensors='pt')
# Indices of input sequence tokens in the vocabulary
print('Input ids:      ', inputs['input_ids'])
# Segment token indices to indicate first and second portions of the inputs. Indices are selected in [0, 1]
print('Token type ids: ', inputs['token_type_ids'])
# Mask to avoid performing attention on padding token indices. Mask values selected in [0, 1]
print('Attention mask: ', inputs['attention_mask'])

Try to find the position of the anwser (part 2)

In [ ]:
# 获取预测的start和end的token的id
answer_id = inputs['input_ids'][0][14 : 19 + 1]
print("answer_id:     ", answer_id)
# 根据id解码出原文
answer_answer = eng_tokenizer.decode(answer_id)
print("answer_answer: ", answer_answer)

Start to Training

In [ ]:
# 指定正确答案的开始结束位置，14
output = model(**inputs, start_positions=torch.tensor([14]), end_positions=torch.tensor([19]))
print("loss: ", output.loss)

optimizer = AdamW(model.parameters(), lr=1e-4)
output.loss.backward()
optimizer.step()

Testing Again

In [ ]:
question = 'Why does Jeanie like Tom?'
paragraph = 'Jeanie likes Tom because Tom is good at deep learning.'
inputs = eng_tokenizer(question, paragraph, return_tensors='pt')

with torch.no_grad():
    output = model(**inputs)
# output = model(input_ids=inputs['input_ids'], token_type_ids=inputs['token_type_ids'], attention_mask=inputs['attention_mask'])

print("start_logits: ")
print(output.start_logits)

print("end_logits: ")
print(output.end_logits)

start = torch.argmax(output.start_logits) # 返回dim维度上张量最大值的索引。
end = torch.argmax(output.end_logits)
print("start position: ", start.item()) # 一个元素张量可以用x.item()得到元素值
print("end position:   ", end.item())

# 获取预测的start和end的token的id
predict_id = inputs['input_ids'][0][start : end + 1]
print("predict_id:     ", predict_id)
# 根据id解码出原文
predict_answer = eng_tokenizer.decode(predict_id)
print("predict_answer: ", predict_answer)

## Other Test Topic

(See https://huggingface.co/docs/accelerate/usage_guides/gradient_accumulation)

In [ ]:
# define toy inputs and labels
x = torch.tensor([1., 2., 3., 4., 5., 6., 7., 8.])
y = torch.tensor([2., 4., 6., 8., 10., 12., 14., 16.])
gradient_accumulation_steps = 4
batch_size = len(x) // gradient_accumulation_steps

print ("len(x):", len(x))
print ("batch_size:", batch_size)